<a href="https://colab.research.google.com/github/Leonardozepeda04/edt-dataa-pipeline/blob/main/notebooks/polizas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [8]:
import numpy as np

In [2]:
#Crear dataset
url_tipos = "https://raw.githubusercontent.com/Leonardozepeda04/edt-dataa-pipeline/refs/heads/main/data/raw/polizas.csv"

In [3]:
polizas = pd.read_csv(url_tipos)

In [5]:
#Exploracion de datos
polizas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25150 entries, 0 to 25149
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_poliza        25150 non-null  int64 
 1   fecha_emision    22739 non-null  object
 2   id_cliente       25150 non-null  int64 
 3   id_corredor      25150 non-null  int64 
 4   id_aseguradora   25150 non-null  int64 
 5   id_tipo_seguro   25150 non-null  int64 
 6   prima            21840 non-null  object
 7   comision         21715 non-null  object
 8   monto_asegurado  21787 non-null  object
dtypes: int64(5), object(4)
memory usage: 1.7+ MB


In [6]:
#Limpieza de datos
def limpiar_dataframe(df):

    df.columns = df.columns.str.strip().str.lower()

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()

    df = df.replace(r'^\s*$', pd.NA, regex=True)

    df = df.drop_duplicates()

    return df

In [9]:
# Transformaciones

# --- Normalizar espacios ---
polizas['fecha_emision'] = polizas['fecha_emision'].astype(str).str.strip()

# --- Función para limpiar columnas numéricas mixtas ---
def limpiar_numero(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()

    if valor in ["", "-", "nan", "None"]:
        return np.nan

    # Caso 1: formato europeo 27.544,32
    if "." in valor and "," in valor:
        if valor.rfind(",") > valor.rfind("."):
            valor = valor.replace(".", "").replace(",", ".")
        else:
            valor = valor.replace(",", "")

    # Caso 2: solo coma 829,53
    elif "," in valor:
        valor = valor.replace(",", ".")

    # Caso 3: solo punto -> se deja igual
    try:
        return float(valor)
    except:
        return np.nan

# Aplicar limpieza a columnas numéricas
polizas['prima'] = polizas['prima'].apply(limpiar_numero)
polizas['comision'] = polizas['comision'].apply(limpiar_numero)
polizas['monto_asegurado'] = polizas['monto_asegurado'].apply(limpiar_numero)

# --- Limpieza de fecha ---
polizas['fecha_emision'] = polizas['fecha_emision'].replace(['nan', 'None', '-'], pd.NA)

polizas['fecha_emision'] = pd.to_datetime(
    polizas['fecha_emision'],
    errors='coerce'
)

# --- Columnas derivadas ---
polizas['anio'] = polizas['fecha_emision'].dt.year
polizas['mes'] = polizas['fecha_emision'].dt.month

In [10]:
#Ver resultados
print(polizas)

       id_poliza fecha_emision  id_cliente  id_corredor  id_aseguradora  \
0              1           NaT         184           42              15   
1              2    2026-02-16        2408           35              11   
2              3           NaT         540           42               4   
3              4           NaT        2821           40              10   
4              5           NaT         945           23               9   
...          ...           ...         ...          ...             ...   
25145      25146           NaT        2328           27               3   
25146      25147    2025-07-02        2295           16               1   
25147      25148           NaT         133           48              10   
25148      25149           NaT        2221           29              13   
25149      25150           NaT        3676           54               1   

       id_tipo_seguro    prima  comision  monto_asegurado    anio  mes  
0                   2   82

In [11]:
# --- Separar válidos y rechazados ---
validos = polizas[
    polizas['id_poliza'].notna() &
    polizas['fecha_emision'].notna() &
    polizas['id_cliente'].notna() &
    polizas['id_corredor'].notna() &
    polizas['id_aseguradora'].notna() &
    polizas['id_tipo_seguro'].notna() &
    polizas['prima'].notna() &
    polizas['comision'].notna() &
    polizas['monto_asegurado'].notna()
].copy()

rechazados = polizas[
    polizas['id_poliza'].isna() |
    polizas['fecha_emision'].isna() |
    polizas['id_cliente'].isna() |
    polizas['id_corredor'].isna() |
    polizas['id_aseguradora'].isna() |
    polizas['id_tipo_seguro'].isna() |
    polizas['prima'].isna() |
    polizas['comision'].isna() |
    polizas['monto_asegurado'].isna()
].copy()

In [12]:
#Mostrar resultados
print("✅ Válidos:")
print(validos)

print("\n❌ Rechazados:")
print(rechazados)

✅ Válidos:
       id_poliza fecha_emision  id_cliente  id_corredor  id_aseguradora  \
9             10    2026-01-24        2281           69              13   
25            26    2025-07-29        2295           71              11   
44            45    2025-12-11        2278           66               3   
48            49    2025-08-11        1122            2              14   
49            50    2025-02-22         717           49              15   
...          ...           ...         ...          ...             ...   
25118      25119    2025-11-02        2655          116               6   
25124      25125    2026-01-10        1589           62              11   
25126      25127    2025-03-02          18           80              15   
25138      25139    2025-06-07        2312           20              15   
25144      25145    2025-05-18        1250           61               8   

       id_tipo_seguro    prima  comision  monto_asegurado    anio   mes  
9             

In [13]:
# --- Motivos de rechazo ---
def motivo(row):
    motivos = []

    if pd.isna(row['id_poliza']):
        motivos.append("id_poliza_vacio")
    if pd.isna(row['fecha_emision']):
        motivos.append("fecha_emision_vacia")
    if pd.isna(row['id_cliente']):
        motivos.append("id_cliente_vacio")
    if pd.isna(row['id_corredor']):
        motivos.append("id_corredor_vacio")
    if pd.isna(row['id_aseguradora']):
        motivos.append("id_aseguradora_vacia")
    if pd.isna(row['id_tipo_seguro']):
        motivos.append("id_tipo_seguro_vacio")
    if pd.isna(row['prima']):
        motivos.append("prima_vacia")
    elif row['prima'] <= 0:
        motivos.append("prima_invalida")
    if pd.isna(row['comision']):
        motivos.append("comision_vacia")
    elif row['comision'] < 0:
        motivos.append("comision_invalida")
    if pd.isna(row['monto_asegurado']):
        motivos.append("monto_asegurado_vacio")
    elif row['monto_asegurado'] <= 0:
        motivos.append("monto_asegurado_invalido")

    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

In [14]:
print("\n❌ Rechazados con motivos:")
display(rechazados[
    [
        'id_poliza',
        'fecha_emision',
        'id_cliente',
        'id_corredor',
        'id_aseguradora',
        'id_tipo_seguro',
        'prima',
        'comision',
        'monto_asegurado',
        'motivo_rechazo'
    ]
])


❌ Rechazados con motivos:


,id_poliza,fecha_emision,id_cliente,id_corredor,id_aseguradora,id_tipo_seguro,prima,comision,monto_asegurado,motivo_rechazo
0,1,NaT,184,42,15,2,829.53,NaN,139253.11,"fecha_emision_vacia,comision_vacia"
1,2,2026-02-16,2408,35,11,12,NaN,12.22,27544.32,prima_vacia
2,3,NaT,540,42,4,9,1611.53,92.05,173298.36,fecha_emision_vacia
3,4,NaT,2821,40,10,5,1866.62,456.99,244461.27,fecha_emision_vacia
4,5,NaT,945,23,9,11,NaN,324.08,123407.75,"fecha_emision_vacia,prima_vacia"
...,...,...,...,...,...,...,...,...,...,...
25145,25146,NaT,2328,27,3,6,1981.61,3190.16,377247.62,fecha_emision_vacia
25146,25147,2025-07-02,2295,16,1,9,1997.65,242.32,NaN,monto_asegurado_vacio
25147,25148,NaT,133,48,10,4,171.49,18.12,6915.63,fecha_emision_vacia
25148,25149,NaT,2221,29,13,3,NaN,226.04,169766.06,"fecha_emision_vacia,prima_vacia"


In [15]:
# Exportar archivos
polizas.to_csv("polizas_curated.csv", index=False)
validos.to_csv("polizas_validos.csv", index=False)
rechazados.to_csv("polizas_rechazados.csv", index=False)